In [9]:
import pandas as pd
import pickle

from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import root_mean_squared_error

import mlflow

In [10]:

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment('nyc-taxi-experiment')

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1787300277912, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1787300277912, lifecycle_stage='active', name='nyc-taxi-experiment', tags={}, trace_location=None, workspace='default'>

In [13]:

def read_dataframe(filename):
    df = pd.read_parquet(filename)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime 
    df['duration'] = df['duration'].apply(lambda td: td.total_seconds() / 60)

    df = df[((df.duration >=1) & (df.duration <=60))]

    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)

    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']

    return df

In [14]:
train_file = 'https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-01.parquet'
val_file = 'https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2021-02.parquet'

df_train = read_dataframe(train_file)
df_val = read_dataframe(val_file)

In [17]:
categorical = ['PU_DO']
numerical = ['trip_distance']

dv = DictVectorizer()

train_dicts = df_train[categorical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

target = 'duration'
y_train = df_train[target].values
y_val = df_val[target].values

In [18]:
import xgboost as xgb
from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope

In [21]:
from pathlib import Path
models_folder = Path('models')
models_folder.mkdir(exist_ok=True)

In [22]:
mlflow.xgboost.autolog(disable=True)

with mlflow.start_run():
    
    train = xgb.DMatrix(X_train, label=y_train)
    valid = xgb.DMatrix(X_val, label=y_val)

    best_params = {
        'learning_rate': 0.21522370937264673,
        'max_depth': 26,
        'min_child_weight': 4.107367764834471,
        'objective': 'reg:linear',
        'reg_alpha': 0.09099402339275377,
        'reg_lambda': 0.04294359447106556,
        'seed': 42
    }

    mlflow.log_params(best_params)

    booster = xgb.train(
        params=best_params,
        dtrain=train,
        num_boost_round=1000,
        evals=[(valid, 'validation')],
        early_stopping_rounds=50
    )

    y_pred = booster.predict(valid)
    rmse = root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)

    with open("models/preprocessor.b", "wb") as f_out:
        pickle.dump(dv, f_out)
    mlflow.log_artifact("models/preprocessor.b", artifact_path="preprocessor")

    mlflow.xgboost.log_model(booster, artifact_path="models_mlflow")

[0]	validation-rmse:11.93369
[1]	validation-rmse:11.73819
[2]	validation-rmse:11.60874
[3]	validation-rmse:11.51659


/opt/anaconda3/envs/interview_prep/lib/python3.11/site-packages/xgboost/callback.py:385: UserWarning: [12:23:42] WARNING: /Users/runner/work/xgboost/xgboost/src/objective/regression_obj.cu:277: reg:linear is now deprecated in favor of reg:squarederror.
  self.starting_round = model.num_boosted_rounds()


[4]	validation-rmse:11.44232
[5]	validation-rmse:11.38288
[6]	validation-rmse:11.32999
[7]	validation-rmse:11.28328
[8]	validation-rmse:11.23386
[9]	validation-rmse:11.20111
[10]	validation-rmse:11.16075
[11]	validation-rmse:11.12956
[12]	validation-rmse:11.09616
[13]	validation-rmse:11.06494
[14]	validation-rmse:11.03267
[15]	validation-rmse:11.00298
[16]	validation-rmse:10.96981
[17]	validation-rmse:10.94203
[18]	validation-rmse:10.91522
[19]	validation-rmse:10.89187
[20]	validation-rmse:10.86629
[21]	validation-rmse:10.85033
[22]	validation-rmse:10.82650
[23]	validation-rmse:10.80697
[24]	validation-rmse:10.78635
[25]	validation-rmse:10.76483
[26]	validation-rmse:10.74349
[27]	validation-rmse:10.72431
[28]	validation-rmse:10.70387
[29]	validation-rmse:10.68565
[30]	validation-rmse:10.66702
[31]	validation-rmse:10.64708
[32]	validation-rmse:10.63085
[33]	validation-rmse:10.61668
[34]	validation-rmse:10.59857
[35]	validation-rmse:10.58358
[36]	validation-rmse:10.56520
[37]	validation-

2026/08/21 12:24:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


🏃 View run welcoming-steed-14 at: http://127.0.0.1:5000/#/experiments/1/runs/2927fcbca9204efa8805d9a60110e61b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1
